# SAE / VUF MUC (Colab)

Пайплайн `sae_muc.run_muc` с двумя режимами интервенции:

- **`sae`** — латентный bump: **encode → f + αδ → decode + error** (нужен `mistral_intervention.pt` из `build_intervention_config`).
- **`residual`** — корректировка как в статье: **h ← h + α·r̂** по строкам `Hs_hedge` (сырой остаток, без SAE на инференсе).

**Данные в `REPO_DIR`** (по умолчанию `/content/sae-muc`):

- `datasets/{dataset}/{model}/{split}.csv`
- `detection/LR_outputs/{dataset}/{model}/{split}_verbal_uncertainty_sentence_semantic_entropy.json`
- `Hs_hedge_universal.pt` — для **residual** и для сборки SAE-конфига (см. §6–§7, путь задаётся в ячейке).

**Простой старт:** §2 — загрузка **одного zip** с папками `datasets/`, `detection/`, `calibration/` в корне архива. Либо скопируйте с Drive (§3–§4).

**Dry run:** в §6 задайте `DRY_RUN_N = 5` (первые N вопросов); для полного прогона — `None`. Прогоны SAE и residual — **две ячейки** §7 и §8.

**Нужны:** GPU, Hugging Face токен для Mistral. Если сессия «засорена» импортами: **Runtime → Restart runtime**, затем ячейки по порядку.

## 0. GPU (без `import torch`)

**Важно:** не импортируйте `torch` до ячейки с `pip` — иначе в процессе залипает старый NumPy и дальше `transformers` падает с `dtype size changed`.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 1. Клон репозитория и зависимости

**Время:** клон — секунды; **pip** (особенно **transformers** с `--force-reinstall`) — нередко **3–10 минут**. Лог pip в ячейке полный, без `-q`.

In [ ]:
import os, sys, subprocess

GIT_URL = os.environ.get("SAE_MUC_GIT_URL", "https://github.com/SadreevAmir/sae-muc.git")
GIT_BRANCH = os.environ.get("SAE_MUC_BRANCH", "main")
REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    parent = os.path.dirname(REPO_DIR.rstrip("/")) or "/content"
    os.makedirs(parent, exist_ok=True)
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

assert os.path.isdir(sae_pkg), f"После клона ожидается {sae_pkg}"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# PyTorch не трогаем. Сброс NumPy + совместимые бинари transformers (иначе dtype size changed).
print("→ pip uninstall numpy …")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"])
print("→ pip install numpy (force-reinstall) …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", "--force-reinstall",
    "numpy>=2.0.0,<2.1",
])
print("→ pip install transformers + accelerate (force-reinstall, может быть долго) …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", "--force-reinstall",
    "transformers>=4.40", "accelerate",
])
print("→ pip install sae-lens и остальное …")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U",
    "sae-lens>=6.0", "pandas", "tqdm", "jsonlines", "huggingface_hub",
])
subprocess.check_call([
    sys.executable, "-c",
    "import numpy, numpy.random; print('numpy OK', numpy.__version__)",
])

import torch

assert torch.cuda.is_available(), "Runtime → тип подключения: GPU"
print("REPO_DIR:", REPO_DIR)
print("GPU:", torch.cuda.get_device_name(0))


## 2. Данные VUF: один zip → `REPO_DIR`

Соберите архив так, чтобы **в корне zip** были каталоги `datasets/`, `detection/`, `calibration/` (как в чекпоинте фаз 1–6). Нужен **`Hs_hedge_universal.pt`**: полный чекпоинт — часто `merged/.../uncertainty/` и/или `merged/.../sentence/`; **без uncertainty-промпта** используйте zip только с `calibration/outputs/merged/.../sentence/Hs_hedge_universal.pt` (локально: `vuf_checkpoint/build_colab_bundle.py --no-uncertainty`).

Установите `DO_UPLOAD_ZIP = True` и выполните ячейку — выберите файл. Если данные уже на месте (Drive, предыдущий запуск), поставьте `False`.

In [ ]:
import os
import zipfile

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

# True — диалог загрузки zip в Colab
DO_UPLOAD_ZIP = False

if DO_UPLOAD_ZIP:
    try:
        from google.colab import files
        print("Загрузите zip (в корне: datasets/, detection/, calibration/) …")
        uploaded = files.upload()
        for fn in uploaded:
            if str(fn).lower().endswith(".zip"):
                with zipfile.ZipFile(fn, "r") as z:
                    z.extractall(REPO_DIR)
                print("Распаковано в", REPO_DIR)
                break
        else:
            print("Нет .zip среди загруженных файлов.")
    except ImportError:
        print("Не Colab — распакуйте данные в REPO_DIR вручную.")
else:
    print("DO_UPLOAD_ZIP=False — ожидаются данные уже в REPO_DIR.")

## 3. Google Drive (опционально): монтирование и бэкап

Поставьте `MOUNT_DRIVE = False`, если данные уже в `REPO_DIR` и бэкап не нужен. Иначе смонтируйте Drive и задайте `DRIVE_BACKUP_ROOT`; каждые `BACKUP_INTERVAL_SEC` секунд копируются `sae_muc/outputs` и `sae_muc/artifacts`.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

MOUNT_DRIVE = False  # True — смонтировать Drive и фоновый бэкап
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/vuf_sae_muc_backup"
BACKUP_INTERVAL_SEC = 600

_drive_backup = None
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        from sae_muc.drive_sync import PeriodicDriveBackup

        drive.mount("/content/drive")
        pairs = [
            (f"{REPO_DIR}/sae_muc/outputs", f"{DRIVE_BACKUP_ROOT}/sae_muc/outputs"),
            (f"{REPO_DIR}/sae_muc/artifacts", f"{DRIVE_BACKUP_ROOT}/sae_muc/artifacts"),
        ]
        Path(DRIVE_BACKUP_ROOT).mkdir(parents=True, exist_ok=True)
        _drive_backup = PeriodicDriveBackup(pairs, interval_sec=BACKUP_INTERVAL_SEC)
        _drive_backup.start()
        print("Периодический бэкап на Drive запущен. Остановка: _drive_backup.stop()")
    except ImportError:
        print("Не Colab — пропуск Drive.")
else:
    print("MOUNT_DRIVE=False — без монтирования Drive.")

## 4. (Опционально) Чекпоинт фаз 1–6 с Drive → `REPO_DIR`

Если данные не заливали zip в §2, а лежат на Drive в папке вроде `vuf_checkpoint_phase6`, скопируйте `datasets`, `detection`, `calibration` в `REPO_DIR`.

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR = Path(os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc"))
CHECKPOINT_SRC = Path("/content/drive/MyDrive/vuf_checkpoint_phase6")  # или None

if CHECKPOINT_SRC and CHECKPOINT_SRC.is_dir():
    for name in ("datasets", "detection", "calibration"):
        src = CHECKPOINT_SRC / name
        dst = REPO_DIR / name
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("Copied", name)
else:
    print("Пропуск копирования чекпоинта (нет CHECKPOINT_SRC).")

## 5. Hugging Face login

In [ ]:
from huggingface_hub import login
login()  # токен с доступом к Mistral

## 6. Конфиг: dry run, `Hs_hedge`, сборка `mistral_intervention.pt`

Здесь задаются **`PROMPT_TYPE`**, **`DRY_RUN_N`**, **`MAX_ALPHA`**, **`ALPHA_STEP`** (кратность α после clip, напр. шаг **5** при `MAX_ALPHA=20` → значения 0, 5, 10, 15, 20), **`GEN_BATCH_SIZE`**, **`STR_PROCESS_LAYERS`**, **`Hs_hedge`**, сборка **`mistral_intervention.pt`**.

**`PROMPT_TYPE`**: `"sentence"` / `"plain"` / `"uncertainty"` — см. интро; суффиксы в имени jsonl: `_sentence` / `_plain` / без суффикса для uncertainty.

**Почему при `alpha == 0` в jsonl пустые ответы:** в `run_muc` для `alpha == 0` или `detection == 0` **намеренно не вызывается** `model.generate`: в файл пишется заглушка (`most_likely_answer` и `responses` пустые), чтобы не тратить GPU на строки без интервенции (как в исходной логике Meta / детекции).

In [ ]:
import os
import subprocess
import sys

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")

# Тест на первых N строках CSV; None — весь split
DRY_RUN_N = 5
# Верхняя граница α и квантование до кратных ALPHA_STEP (0.0 = как в CLI: round 4 знака)
MAX_ALPHA = 20.0
ALPHA_STEP = 5.0

# GPU-батч: подряд идущие строки с одинаковыми α и detection — одним generate (A100/L4: 16–32; при OOM уменьшить)
GEN_BATCH_SIZE = 16

# Какие слои участвуют в α / детекции (как в Meta); SAE вешается только на подмножество (есть SAE)
STR_PROCESS_LAYERS = "range(15,32)"

# "uncertainty" | "plain" | "sentence" — см. текст §6 (zip без uncertainty hedge → "sentence" или "plain")
PROMPT_TYPE = "sentence"

# Только residual: None → авто (пересечение с SAE-слоями); или явно "15,23" / "range(15,32)"
VUF_LAYERS = None
VUF_ALIGN_RELEASE = "mistral-7b-res-wg"  # если нет intervention.pt, откуда брать список HF-слоёв SAE

MN = "Mistral-7B-Instruct-v0.3"
_merged_unc = f"{REPO_DIR}/calibration/outputs/merged/{MN}/uncertainty/Hs_hedge_universal.pt"
_merged_sent = f"{REPO_DIR}/calibration/outputs/merged/{MN}/sentence/Hs_hedge_universal.pt"
_nq_unc = f"{REPO_DIR}/calibration/outputs/nq_open/{MN}/uncertainty/Hs_hedge_universal.pt"
if PROMPT_TYPE == "sentence":
    HEDGE_CANDIDATES = [_merged_sent, _merged_unc, _nq_unc]
else:
    HEDGE_CANDIDATES = [_merged_unc, _merged_sent, _nq_unc]
HEDGE_PATH = next((p for p in HEDGE_CANDIDATES if os.path.isfile(p)), None)
if HEDGE_PATH is None:
    raise FileNotFoundError(
        "Не найден Hs_hedge_universal.pt. Проверьте §2/§4 или добавьте путь в HEDGE_CANDIDATES."
    )
print("HEDGE_PATH:", HEDGE_PATH)

OUT_INTERVENTION = f"{REPO_DIR}/sae_muc/artifacts/mistral_intervention.pt"

os.makedirs(os.path.dirname(OUT_INTERVENTION), exist_ok=True)
cmd = [
    sys.executable,
    "-m",
    "sae_muc.build_intervention_config",
    "--hedge_path",
    HEDGE_PATH,
    "--out_path",
    OUT_INTERVENTION,
    "--release",
    "mistral-7b-res-wg",
    "--top_k",
    "64",
    "--sae_device",
    "cpu",
]
subprocess.check_call(cmd, cwd=REPO_DIR)
print("Собрано:", OUT_INTERVENTION)

## 7. Прогон **только SAE** (`--steering sae`)

Латентный bump через SAE. После окончания ячейка **скачивает** готовый `.jsonl` в браузер (Colab), путь также в `RUN_MUC_LAST_JSONL` и в логе `RUN_MUC_LAST_JSONL ...`.

Сначала выполните **§6**. Имя файла: `with_vufi_2_...` **без** `_residual_` в названии.

In [ ]:
import os
import subprocess
import sys

import torch

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")
OUTPUT_DIR = None  # или путь к каталогу для jsonl
AUTO_DOWNLOAD_JSONL = True  # False — только путь в логе, без скачивания


def _colab_prep_run_muc():
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    br = os.environ.get("SAE_MUC_BRANCH", "main")
    gp = subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", br],
        capture_output=True,
        text=True,
    )
    if gp.stdout:
        print(gp.stdout, end="")
    if gp.stderr:
        print(gp.stderr, end="", file=sys.stderr)
    rmp = os.path.join(REPO_DIR, "sae_muc", "run_muc.py")
    with open(rmp, encoding="utf-8") as f:
        src = f.read()
    if "--steering" not in src or "max_questions" not in src:
        raise RuntimeError(f"В {rmp} старая версия run_muc — обновите репозиторий.")
    if "RUN_MUC_LAST_JSONL" not in src:
        print("Замечание: старый run_muc (нет RUN_MUC_LAST_JSONL) — скачивание через последний jsonl в outputs/.")


def _run_muc_cmd(steering: str) -> list:
    c = [
        sys.executable,
        "-m",
        "sae_muc.run_muc",
        "--repo_root",
        REPO_DIR,
        "--dataset",
        "nq_open",
        "--split",
        "test",
        "--model_name",
        "Mistral-7B-Instruct-v0.3",
        "--prompt_type",
        PROMPT_TYPE,
        "--str_process_layers",
        STR_PROCESS_LAYERS,
        "--steering",
        steering,
        "--max_alpha",
        str(float(MAX_ALPHA)),
    ]
    if ALPHA_STEP > 0:
        c += ["--alpha_step", str(float(ALPHA_STEP))]
    if steering == "sae":
        c += ["--intervention_path", OUT_INTERVENTION]
    else:
        c += ["--hedge_path", HEDGE_PATH, "--intervention_path", OUT_INTERVENTION]
        c += ["--vuf_align_release", VUF_ALIGN_RELEASE]
        if VUF_LAYERS:
            c += ["--vuf_layers", VUF_LAYERS]
    if DRY_RUN_N is not None:
        c += ["--max_questions", str(int(DRY_RUN_N))]
    c += ["--gen_batch_size", str(int(GEN_BATCH_SIZE))]
    if OUTPUT_DIR:
        c += ["--output_dir", OUTPUT_DIR]
    return c


def _invoke_run_muc(steering: str):
    print("CLI:", " ".join(_run_muc_cmd(steering)))
    for name in list(sys.modules):
        if name == "sae_muc" or name.startswith("sae_muc."):
            del sys.modules[name]
    cli = _run_muc_cmd(steering)[3:]
    saved = sys.argv
    try:
        sys.argv = ["sae_muc.run_muc"] + cli
        from sae_muc.run_muc import main as main_run_muc

        main_run_muc()
    finally:
        sys.argv = saved
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _guess_jsonl_path(steering: str):
    out_dir = os.path.join(REPO_DIR, "sae_muc", "outputs", "nq_open", MN, PROMPT_TYPE, "test")
    if not os.path.isdir(out_dir):
        return None
    cand = [
        os.path.join(out_dir, f)
        for f in os.listdir(out_dir)
        if f.endswith(".jsonl") and f.startswith("with_vufi_")
    ]
    if not cand:
        return None
    if steering == "sae":
        cand = [p for p in cand if "_residual" not in os.path.basename(p)]
    else:
        cand = [p for p in cand if "_residual" in os.path.basename(p)]
    if not cand:
        return None
    return max(cand, key=os.path.getmtime)


def _download_last_jsonl(steering: str):
    p = os.environ.get("RUN_MUC_LAST_JSONL")
    if not p or not os.path.isfile(p):
        p = _guess_jsonl_path(steering)
    if not p or not os.path.isfile(p):
        print("jsonl не найден. Проверьте каталог:", os.path.join(REPO_DIR, "sae_muc", "outputs", "nq_open", MN, PROMPT_TYPE, "test"))
        return
    print("Готовый jsonl:", p)
    if not AUTO_DOWNLOAD_JSONL:
        return
    try:
        from google.colab import files

        files.download(p)
        print("Скачивание в браузер запущено (Colab).")
    except ImportError:
        print("Не Colab — скопируйте файл вручную по пути выше.")


_colab_prep_run_muc()
_invoke_run_muc("sae")
_download_last_jsonl("sae")

## 8. Прогон **только residual VUF** (`--steering residual`)

Стандартный **h ← h + α·r̂** по `Hs_hedge`. При **`VUF_LAYERS = None` в §6** интервенция вешается **только на те HF-слои, для которых есть SAE** в `mistral_intervention.pt` (пересечение с `STR_PROCESS_LAYERS`), как в коде `run_muc`.

Выполните **§6** (и при необходимости §7 — можно без него, если `mistral_intervention.pt` уже собран). После прогона — **скачивание** `.jsonl` (имя с суффиксом **`_residual_L…`**).

In [ ]:
import os
import subprocess
import sys

import torch

REPO_DIR = os.environ.get("SAE_MUC_REPO_DIR", "/content/sae-muc")
OUTPUT_DIR = None
AUTO_DOWNLOAD_JSONL = True


def _colab_prep_run_muc():
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    br = os.environ.get("SAE_MUC_BRANCH", "main")
    gp = subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", br],
        capture_output=True,
        text=True,
    )
    if gp.stdout:
        print(gp.stdout, end="")
    if gp.stderr:
        print(gp.stderr, end="", file=sys.stderr)
    rmp = os.path.join(REPO_DIR, "sae_muc", "run_muc.py")
    with open(rmp, encoding="utf-8") as f:
        src = f.read()
    if "--steering" not in src or "max_questions" not in src:
        raise RuntimeError(f"В {rmp} старая версия run_muc — обновите репозиторий.")
    if "RUN_MUC_LAST_JSONL" not in src:
        print("Замечание: старый run_muc (нет RUN_MUC_LAST_JSONL) — скачивание через последний jsonl в outputs/.")


def _run_muc_cmd(steering: str) -> list:
    c = [
        sys.executable,
        "-m",
        "sae_muc.run_muc",
        "--repo_root",
        REPO_DIR,
        "--dataset",
        "nq_open",
        "--split",
        "test",
        "--model_name",
        "Mistral-7B-Instruct-v0.3",
        "--prompt_type",
        PROMPT_TYPE,
        "--str_process_layers",
        STR_PROCESS_LAYERS,
        "--steering",
        steering,
        "--max_alpha",
        str(float(MAX_ALPHA)),
    ]
    if ALPHA_STEP > 0:
        c += ["--alpha_step", str(float(ALPHA_STEP))]
    if steering == "sae":
        c += ["--intervention_path", OUT_INTERVENTION]
    else:
        c += ["--hedge_path", HEDGE_PATH, "--intervention_path", OUT_INTERVENTION]
        c += ["--vuf_align_release", VUF_ALIGN_RELEASE]
        if VUF_LAYERS:
            c += ["--vuf_layers", VUF_LAYERS]
    if DRY_RUN_N is not None:
        c += ["--max_questions", str(int(DRY_RUN_N))]
    c += ["--gen_batch_size", str(int(GEN_BATCH_SIZE))]
    if OUTPUT_DIR:
        c += ["--output_dir", OUTPUT_DIR]
    return c


def _invoke_run_muc(steering: str):
    print("CLI:", " ".join(_run_muc_cmd(steering)))
    for name in list(sys.modules):
        if name == "sae_muc" or name.startswith("sae_muc."):
            del sys.modules[name]
    cli = _run_muc_cmd(steering)[3:]
    saved = sys.argv
    try:
        sys.argv = ["sae_muc.run_muc"] + cli
        from sae_muc.run_muc import main as main_run_muc

        main_run_muc()
    finally:
        sys.argv = saved
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _guess_jsonl_path(steering: str):
    out_dir = os.path.join(REPO_DIR, "sae_muc", "outputs", "nq_open", MN, PROMPT_TYPE, "test")
    if not os.path.isdir(out_dir):
        return None
    cand = [
        os.path.join(out_dir, f)
        for f in os.listdir(out_dir)
        if f.endswith(".jsonl") and f.startswith("with_vufi_")
    ]
    if not cand:
        return None
    if steering == "sae":
        cand = [p for p in cand if "_residual" not in os.path.basename(p)]
    else:
        cand = [p for p in cand if "_residual" in os.path.basename(p)]
    if not cand:
        return None
    return max(cand, key=os.path.getmtime)


def _download_last_jsonl(steering: str):
    p = os.environ.get("RUN_MUC_LAST_JSONL")
    if not p or not os.path.isfile(p):
        p = _guess_jsonl_path(steering)
    if not p or not os.path.isfile(p):
        print("jsonl не найден. Проверьте каталог:", os.path.join(REPO_DIR, "sae_muc", "outputs", "nq_open", MN, PROMPT_TYPE, "test"))
        return
    print("Готовый jsonl:", p)
    if not AUTO_DOWNLOAD_JSONL:
        return
    try:
        from google.colab import files

        files.download(p)
        print("Скачивание в браузер запущено (Colab).")
    except ImportError:
        print("Не Colab — скопируйте файл вручную по пути выше.")


_colab_prep_run_muc()
_invoke_run_muc("residual")
_download_last_jsonl("residual")

## 9. Остановить бэкап (опционально)

In [ ]:
# if _drive_backup is not None:
#     _drive_backup.stop()